# Figure 1: scPLAD framework schematic

Reproduce and audit the manuscript figure from archived results. Model training is documented but is **not executed**.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import Markdown, SVG, display

HERE = Path.cwd().resolve()
ARCHIVE = HERE.parent if HERE.name == "notebooks" else HERE
if not (ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv").exists():
    raise FileNotFoundError("Run this notebook from the archive root or notebooks directory.")

REGISTRY = pd.read_csv(
    ARCHIVE / "manifests" / "PANEL_REPRODUCIBILITY.tsv",
    sep="\t",
    keep_default_na=False,
)

default_config = ARCHIVE / "reproducibility" / "configs" / "provided_results.json"
config_path = Path(os.environ.get("SCPLAD_REPRO_CONFIG", default_config)).expanduser().resolve()
REPRO_CONFIG = json.loads(config_path.read_text(encoding="utf-8"))
config_dir = config_path.parent

def resolve_config_path(key, fallback):
    env_key = {
        "source_data_root": "SCPLAD_SOURCE_DATA_ROOT",
        "reproduced_root": "SCPLAD_REPRODUCED_ROOT",
    }[key]
    value = os.environ.get(env_key, REPRO_CONFIG.get("paths", {}).get(key, fallback))
    path = Path(value).expanduser()
    return path if path.is_absolute() else (config_dir / path).resolve()

SOURCE_DATA = resolve_config_path("source_data_root", str(ARCHIVE / "source_data"))
REPRO = resolve_config_path("reproduced_root", str(ARCHIVE / "reproduced"))
REPRO.mkdir(parents=True, exist_ok=True)
os.environ["SCPLAD_REPRO_CONFIG"] = str(config_path)
os.environ["SCPLAD_SOURCE_DATA_ROOT"] = str(SOURCE_DATA)
os.environ["SCPLAD_REPRODUCED_ROOT"] = str(REPRO)
print(f"Figure archive: {ARCHIVE}")
print(f"Input mode: {REPRO_CONFIG['mode']}")
print(f"Source data: {SOURCE_DATA}")
print(f"Output root: {REPRO}")

## Experiment and implementation provenance

Figure 1 is a scientific schematic rather than a numerical result.
Panels map the cross-cell task, frozen PatchAE tokenizer, biological-prior
condition encoder, and control-anchored displacement diffusion to the
registered training/inference entry points. The notebook verifies the
editable assets instead of fabricating quantitative source data.

In [ ]:
panel_map = REGISTRY.loc[REGISTRY["figure"].eq("Fig1")].copy()
required = ["source_data", "training_code", "evaluation_code", "plot_code", "canonical_panel"]
display(panel_map[["panel", "panel_type", "experiment_id", "claim_or_role", "status"]])

def archived_paths_exist(value, base):
    if value == "NA":
        return True
    return all((base / item).exists() for item in str(value).split(";"))

for column in ["source_data", "plot_code", "canonical_panel"]:
    missing = [
        value for value in panel_map[column]
        if not archived_paths_exist(value, ARCHIVE)
    ]
    assert not missing, f"Missing {column}: {missing}"
print("Panel-level figure inputs and plotting assets are present.")

In [ ]:
subprocess.run(
    [sys.executable, str(ARCHIVE / "scripts" / "Fig1" / "export_figure1_subpanels.py")],
    check=True,
)
outputs = [
    ("a", "Cross-cell-line task and information flow", REPRO / "Fig1" / "figure1a_standalone.svg"),
    ("b", "Frozen pathway-aware latent tokenizer", REPRO / "Fig1" / "figure1b_standalone.svg"),
    ("c", "Multi-source biological-prior encoder", REPRO / "Fig1" / "figure1c_standalone.svg"),
    ("d", "Control-anchored latent displacement diffusion", REPRO / "Fig1" / "figure1d_standalone.svg"),
]
assert all(path.exists() for _, _, path in outputs)
for panel, title, path in outputs:
    display(Markdown(f"### Fig. 1{panel}: {title}"))
    display(SVG(filename=str(path)))